# FakeGuard — AI Fake News Detector
### NeuroLogic '26 Datathon

> **How to use:** Run cells one at a time, top to bottom.
> Each cell does ONE thing. If a cell fails, fix it and re-run — no need to go back to Cell 1.

---

### Errors fixed in this version
| Error | Fix Applied |
|-------|-------------|
| `peft` conflicts with `transformers` | Cell 1 uninstalls peft |
| P100 CUDA sm_60 not supported by PyTorch 2.10 | Cell 3 warns — use T4 GPU |
| `evaluation_strategy` deprecated | Cell 8 uses `eval_strategy` |
| `Trainer(tokenizer=)` deprecated | Cell 8 omits that argument |
| `PicklingError` from module patching | Cell 8 is fully self-contained |
| Module cache conflict after `git pull` | Cell 8 does NOT import src/train.py |

---

| Cell | Task | Time |
|------|------|------|
| 1 | Uninstall peft, install packages | 30 sec |
| 2 | Restart kernel | instant |
| 3 | GPU check (must be T4, not P100) | instant |
| 4 | Clone GitHub repo | 30 sec |
| 5 | Find dataset | instant |
| 6 | Preprocessing | 2 min |
| 7 | Baseline model (TF-IDF) | 2 min |
| 8 | Train RoBERTa | ~30 min |
| 9 | Evaluate + confusion matrix | 3 min |
| 10 | Generate predictions.csv | 2 min |


## Cell 1 — Fix Package Conflict
Kaggle pre-installs `peft` which conflicts with `transformers`. We uninstall it — we never use it.

In [ ]:
import subprocess, sys

# Uninstall peft — causes ImportError with any transformers version
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'])

# Install only what we need — NO version pins so pip picks compatible set
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers',
    'accelerate',
    'datasets',
    'seaborn',
])

print('Done. Run Cell 2 to restart kernel.')

## Cell 2 — Restart Kernel
Run once. After restart skip Cells 1 and 2 — go straight to Cell 3.

In [ ]:
import IPython
print('Restarting... after restart start from Cell 3.')
IPython.Application.instance().kernel.do_shutdown(restart=True)

## Cell 3 — GPU Check
Must show T4. P100 is NOT compatible with PyTorch 2.10 on Kaggle.
If you see P100: Settings → Accelerator → GPU T4 x1 → Save

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Go to Settings → Accelerator → GPU T4 x1')

gpu_name = torch.cuda.get_device_name(0)
print(f'GPU name: {gpu_name}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

if 'P100' in gpu_name:
    raise RuntimeError('P100 is NOT compatible with PyTorch 2.10. Switch to T4: Settings → Accelerator → GPU T4 x1')

print('GPU is ready.')

## Cell 4 — Clone Repo
Downloads your code from GitHub. If already cloned, just pulls latest changes.

In [ ]:
import subprocess, os, sys

REPO_URL  = 'https://github.com/ayushtiwari18/neurologic-datathon-fakenews.git'
REPO_ROOT = '/kaggle/working/neurologic-datathon-fakenews'

if not os.path.exists(REPO_ROOT):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_ROOT])
else:
    subprocess.check_call(['git', '-C', REPO_ROOT, 'pull'])

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

for d in ['data/raw', 'data/processed', 'outputs', 'models/roberta_fakenews']:
    os.makedirs(f'{REPO_ROOT}/{d}', exist_ok=True)

print(f'Ready. CWD: {os.getcwd()}')

## Cell 5 — Find Dataset
Finds WELFake_Dataset.csv in /kaggle/input/. If not found, click Add Data on the right panel.

In [ ]:
import glob
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
MODEL_SAVE    = Path(f'{REPO_ROOT}/models/roberta_fakenews')
OUTPUTS_DIR   = Path(f'{REPO_ROOT}/outputs')

matches = glob.glob('/kaggle/input/**/WELFake_Dataset.csv', recursive=True)

if matches:
    DATASET_PATH = matches[0]
    print(f'Found: {DATASET_PATH}')
else:
    DATASET_PATH = None
    print('NOT FOUND. Click Add Data → search WELFake → Add')

## Cell 6 — Preprocessing
Splits dataset into train/val/test. Skips if already done.

In [ ]:
import pandas as pd, shutil, sys, os
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

all_exist = all((PROCESSED_DIR / f).exists() for f in ['train.csv','val.csv','test.csv'])

if all_exist:
    print('CSVs already exist. Loading...')
    train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
    val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
    test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')
else:
    if not 'DATASET_PATH' in dir() or DATASET_PATH is None:
        raise ValueError('Run Cell 5 first to find the dataset.')
    shutil.copy2(DATASET_PATH, f'{REPO_ROOT}/data/raw/WELFake_Dataset.csv')
    from src.preprocess import run_preprocessing
    train_df, val_df, test_df = run_preprocessing()

print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')
print('Done.')

## Cell 7 — Baseline Model
TF-IDF + Logistic Regression. ~2 min. Expected accuracy ~94.66%.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['combined'])
X_val   = vectorizer.transform(val_df['combined'])

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, train_df['label'])

baseline_preds = lr.predict(X_val)
baseline_acc   = accuracy_score(val_df['label'], baseline_preds)

print(f'Baseline Accuracy : {baseline_acc:.4f}')
print(classification_report(val_df['label'], baseline_preds, target_names=['FAKE','REAL']))

## Cell 8 — Train RoBERTa (~30 min)

> **IMPORTANT:** This cell is fully self-contained. It does NOT import src/train.py.
> This avoids ALL version-related errors (PicklingError, module cache, deprecated args).
> Do NOT run any patching cells before this. Clean kernel only.

Fixes baked in:
- `eval_strategy` (not `evaluation_strategy`) — transformers >= 4.45
- No `tokenizer=` in Trainer — deprecated in transformers >= 4.46
- No module patching — avoids PicklingError

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — RoBERTa Training (self-contained, no src/ imports)
# ═══════════════════════════════════════════════════════════════
import os, sys, json, torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
MODEL_SAVE    = Path(f'{REPO_ROOT}/models/roberta_fakenews')
OUTPUTS_DIR   = Path(f'{REPO_ROOT}/outputs')
MODEL_SAVE.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset ───────────────────────────────────────────────────
class FakeNewsDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=512):
        self.df         = pd.read_csv(csv_path)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row['combined']),
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        # Never add token_type_ids — RoBERTa does not use them
        return {
            'input_ids':      enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels':         int(row['label']),
        }

# ── Metrics ───────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }

# ── Tokenizer ─────────────────────────────────────────────────
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

# ── Model ─────────────────────────────────────────────────────
print('Loading roberta-base...')
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=2,
    id2label={0: 'FAKE', 1: 'REAL'},
    label2id={'FAKE': 0, 'REAL': 1},
)

# ── Datasets ──────────────────────────────────────────────────
print('Building datasets...')
train_dataset = FakeNewsDataset(PROCESSED_DIR / 'train.csv', tokenizer)
val_dataset   = FakeNewsDataset(PROCESSED_DIR / 'val.csv',   tokenizer)
print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')

# ── TrainingArguments ─────────────────────────────────────────
# eval_strategy = new name (was evaluation_strategy, removed in transformers>=4.45)
training_args = TrainingArguments(
    output_dir=str(MODEL_SAVE),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    logging_steps=100,
    seed=42,
)

# ── Trainer ───────────────────────────────────────────────────
# tokenizer= argument removed — deprecated in transformers>=4.46
# data_collator handles padding correctly without it
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# ── Train ─────────────────────────────────────────────────────
print('Training started — ~30 min on T4...')
print('-' * 50)
train_result = trainer.train()

# ── Evaluate ──────────────────────────────────────────────────
eval_metrics = trainer.evaluate()
roberta_acc = eval_metrics['eval_accuracy']
roberta_f1  = eval_metrics['eval_f1']
print(f'Val Accuracy : {roberta_acc:.4f}')
print(f'Val F1       : {roberta_f1:.4f}')

# ── Save model ────────────────────────────────────────────────
trainer.save_model(str(MODEL_SAVE))
tokenizer.save_pretrained(str(MODEL_SAVE))
print(f'Model saved → {MODEL_SAVE}')

# ── Save metrics JSON (used by Cell 9) ────────────────────────
metrics_out = {
    'final_val_accuracy':      round(roberta_acc, 6),
    'final_val_f1':            round(roberta_f1, 6),
    'train_runtime_seconds':   round(train_result.metrics.get('train_runtime', 0), 2),
}
json.dump(metrics_out, open(OUTPUTS_DIR / 'training_metrics.json', 'w'), indent=2)
print('Metrics saved. Run Cell 9 next.')

## Cell 9 — Evaluate + Confusion Matrix

In [ ]:
import sys, os, json
from pathlib import Path
from IPython.display import Image, display

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.evaluate import run_evaluation

# Use baseline_acc from Cell 7 if available, else use confirmed value
try:
    baseline_acc
except NameError:
    baseline_acc = 0.9466
    print(f'Using confirmed baseline: {baseline_acc}')

eval_report = run_evaluation(
    val_csv_path      = str(PROCESSED_DIR / 'val.csv'),
    baseline_accuracy = baseline_acc,
)

cm_path = f'{REPO_ROOT}/outputs/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(cm_path))

roberta_acc = eval_report['metrics']['accuracy']
print(f'RoBERTa Accuracy  : {roberta_acc:.4f}')
print(f'Baseline Accuracy : {baseline_acc:.4f}')
print(f'Improvement       : +{roberta_acc - baseline_acc:.4f}')

## Cell 10 — Generate predictions.csv
This is your submission file. Download from the Output panel on the right.

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.predict import run_predict

predictions_df = run_predict(test_csv_path=str(PROCESSED_DIR / 'test.csv'))

print(f'Total : {len(predictions_df):,}')
print(f'FAKE  : {(predictions_df["predicted"] == 0).sum():,}')
print(f'REAL  : {(predictions_df["predicted"] == 1).sum():,}')
print('Saved to outputs/predictions.csv')
print('Download from Output panel on the right.')